# 지도 시각화 (Cartographic Visualization)

_“지도를 만드는 것은 인류의 가장 오래된 지적 노력 중 하나이자 가장 복잡한 것 중 하나입니다. 과학적 이론, 그래픽 표현, 지리적 사실 및 실제적 고려 사항이 끝없는 다양한 방식으로 함께 혼합되어 있습니다.”_ &mdash; [H. J. Steward](https://books.google.com/books?id=cVy1Ms43fFYC)

지도 제작(Cartography) &ndash; 지도를 만드는 연구 및 실습 &ndash; 은 수세기에 걸친 발견과 디자인의 풍부한 역사를 가지고 있습니다. 지도 시각화는 지도 제작 기술을 활용하여 지구 표면의 위치, 경로 또는 궤적과 같은 공간 정보를 포함하는 데이터를 전달합니다.

<div style="float: right; margin-left: 1em; margin-top: 1em;"><img width="300px" src="https://gist.githubusercontent.com/jheer/c90d582ef5322582cf4960ec7689f6f6/raw/8dc92382a837ccc34c076f4ce7dd864e7893324a/latlon.png" /></div>

지구를 구로 근사하면 *위도(latitude)*(적도에서 북쪽 또는 남쪽으로의 도 단위 각도)와 *경도(longitude)*(동서 위치를 지정하는 도 단위 각도)의 구면 좌표계를 사용하여 위치를 나타낼 수 있습니다. 이 시스템에서 *평행선(parallel)*은 위도가 일정한 원이고 *자오선(meridian)*은 경도가 일정한 원입니다. [*본초 자오선(prime meridian)*](https://en.wikipedia.org/wiki/Prime_meridian)은 경도 0°에 위치하며 관례에 따라 영국 그리니치의 왕립 천문대를 통과하도록 정의됩니다.

3차원 구를 2차원 평면에 "평평하게" 만들기 위해 (`경도`, `위도`) 쌍을 (`x`, `y`) 좌표에 매핑하는 [투영(projection)](https://en.wikipedia.org/wiki/Map_projection)을 적용해야 합니다. [스케일(scales)](https://github.com/uwdata/visualization-curriculum/blob/master/altair_scales_axes_legends.ipynb)과 유사하게 투영은 데이터 도메인(공간 위치)에서 시각적 범위(픽셀 위치)로 매핑합니다. 그러나 지금까지 보았던 스케일 매핑은 1차원 도메인을 허용하는 반면, 지도 투영은 본질적으로 2차원입니다.

이 노트북에서는 다음을 포함하여 Altair로 지도를 만들고 공간 데이터를 시각화하는 기본 사항을 소개합니다:

- 지리적 특징을 나타내기 위한 데이터 형식,
- 점, 기호 및 단계 구분도(choropleth maps)와 같은 지도 시각화 기술,
- 일반적인 지도 투영법에 대한 검토.

_이 노트북은 [데이터 시각화 커리큘럼](https://github.com/uwdata/visualization-curriculum)의 일부입니다._

In [1]:
import pandas as pd
import altair as alt
from vega_datasets import data

## 지리적 데이터: GeoJSON 및 TopoJSON (Geographic Data: GeoJSON and TopoJSON)

지금까지 우리는 행(레코드)과 열(필드)로 구성된 데이터 테이블에 해당하는 JSON 및 CSV 형식의 데이터 세트만 다루었습니다. 지리적 영역(국가, 주 등)과 궤적(비행 경로, 지하철 노선 등)을 표현하기 위해서는 풍부한 기하학적 구조(geometries)를 지원하도록 설계된 추가 형식이 필요합니다.

[GeoJSON](https://en.wikipedia.org/wiki/GeoJSON)은 특수한 JSON 형식 내에서 지리적 특징을 모델링합니다. GeoJSON `feature`에는 국가 경계를 구성하는 `경도`, `위도` 좌표와 같은 기하학적 데이터뿐만 아니라 추가적인 데이터 속성이 포함될 수 있습니다.

다음은 미국 콜로라도주의 경계에 대한 GeoJSON `feature` 객체입니다:

~~~ json
{
  "type": "Feature",
  "id": 8,
  "properties": {"name": "Colorado"},
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [[-106.32056285448942,40.998675790862656],[-106.19134826714341,40.99813863734313],[-105.27607827344248,40.99813863734313],[-104.9422739227986,40.99813863734313],[-104.05212898774828,41.00136155846029],[-103.57475287338661,41.00189871197981],[-103.38093099236758,41.00189871197981],[-102.65589358559272,41.00189871197981],[-102.62000064466328,41.00189871197981],[-102.052892177978,41.00189871197981],[-102.052892177978,40.74889940428302],[-102.052892177978,40.69733266640851],[-102.052892177978,40.44003613055551],[-102.052892177978,40.3492571857556],[-102.052892177978,40.00333031918079],[-102.04930288388505,39.57414465707943],[-102.04930288388505,39.56823596836465],[-102.0457135897921,39.1331416175485],[-102.0457135897921,39.0466599009048],[-102.0457135897921,38.69751011321283],[-102.0457135897921,38.61478847120581],[-102.0457135897921,38.268861604631],[-102.0457135897921,38.262415762396685],[-102.04212429569915,37.738153927339205],[-102.04212429569915,37.64415206142214],[-102.04212429569915,37.38900413964724],[-102.04212429569915,36.99365914927603],[-103.00046581851544,37.00010499151034],[-103.08660887674611,37.00010499151034],[-104.00905745863294,36.99580776335414],[-105.15404227428235,36.995270609834606],[-105.2222388620483,36.995270609834606],[-105.7175614468747,36.99580776335414],[-106.00829426840322,36.995270609834606],[-106.47490250048605,36.99365914927603],[-107.4224761410235,37.00010499151034],[-107.48349414060355,37.00010499151034],[-108.38081766383978,36.99903068447129],[-109.04483707103458,36.99903068447129],[-109.04483707103458,37.484617466122884],[-109.04124777694163,37.88049961001363],[-109.04124777694163,38.15283644441336],[-109.05919424740635,38.49983761802722],[-109.05201565922046,39.36680339854235],[-109.05201565922046,39.49786885730673],[-109.05201565922046,39.66062637372313],[-109.05201565922046,40.22248895514744],[-109.05201565922046,40.653823231326896],[-109.05201565922046,41.000287251421234],[-107.91779872584989,41.00189871197981],[-107.3183866123281,41.00297301901887],[-106.85895696843116,41.00189871197981],[-106.32056285448942,40.998675790862656]]
    ]
  }
}
~~~

`feature`에는 임의의 수의 데이터 필드를 포함할 수 있는 `properties` 객체와 `geometry` 객체가 포함됩니다. 이 예에서 `geometry`는 주 경계를 구성하는 `[경도, 위도]` 좌표로 이루어진 단일 다각형(Polygon)을 포함합니다. 좌표는 스크롤하면 한동안 계속됩니다.

GeoJSON의 세부 사항에 대해 더 자세히 알아보려면 [공식 GeoJSON 사양](http://geojson.org/)을 참조하거나 [Tom MacWright의 유용한 입문서](https://macwright.org/2015/03/23/geojson-second-bite)를 읽어보세요.

저장 형식으로서 GeoJSON의 한 가지 단점은 중복될 수 있어 파일 크기가 커질 수 있다는 것입니다. 콜로라도주는 다른 6개 주(애리조나와 닿는 모서리를 포함하면 7개 주)와 경계를 공유합니다. 각 주에 대해 별도의 겹치는 좌표 목록을 사용하는 대신, 공유된 경계를 한 번만 인코딩하여 지리적 영역의 *위상(topology)*을 나타내는 것이 더 효율적인 방법입니다. 다행히도 [TopoJSON](https://github.com/topojson/topojson/blob/master/README.md) 형식이 바로 이 역할을 합니다!

세계 국가의 TopoJSON 파일(110m 해상도)을 로드해 보겠습니다:

In [2]:
world = data.world_110m.url
world

'https://vega.github.io/vega-datasets/data/world-110m.json'

In [3]:
world_topo = data.world_110m()

In [4]:
world_topo.keys()

dict_keys(['type', 'transform', 'objects', 'arcs'])

In [5]:
world_topo['type']

'Topology'

In [6]:
world_topo['objects'].keys()

dict_keys(['land', 'countries'])

*위의 `world_topo` TopoJSON 딕셔너리 객체를 검사하여 내용을 확인해 보세요.*

위의 데이터에서 `objects` 속성은 데이터에서 추출할 수 있는 명명된 요소를 나타냅니다: 모든 `countries`(국가)에 대한 기하학적 구조 또는 지구상의 모든 `land`(육지)를 나타내는 단일 다각형입니다. 이들 중 어느 것이든 우리가 시각화할 수 있는 GeoJSON 데이터로 풀 수 있습니다.

TopoJSON은 특수한 형식이므로, 토폴로지에서 추출할 명명된 피처(feature) 객체를 지정하여 TopoJSON 형식을 파싱하도록 Altair에 지시해야 합니다. 다음 코드는 `world` 데이터 세트에서 `countries` 객체에 대한 GeoJSON 피처를 추출하고 싶음을 나타냅니다:

~~~ js
alt.topo_feature(world, 'countries')
~~~

이 `alt.topo_feature` 메서드 호출은 다음과 같은 Vega-Lite JSON으로 확장됩니다:

~~~ json
{
  "values": world,
  "format": {"type": "topojson", "feature": "countries"}
}
~~~

이제 지리적 데이터를 로드할 수 있게 되었으니, 지도를 만들 준비가 되었습니다!

## 지오셰이프 마크 (Geoshape Marks)

지리적 데이터를 시각화하기 위해 Altair는 `geoshape` 마크 유형을 제공합니다. 기본적인 지도를 만들기 위해 `geoshape` 마크를 생성하고 TopoJSON 데이터를 전달할 수 있습니다. 그러면 이 데이터는 전 세계 각 국가에 대해 하나씩 GeoJSON 피처로 풀립니다:

In [7]:
alt.Chart(alt.topo_feature(world, 'countries')).mark_geoshape()

alt.Chart(...)

위의 예제에서 Altair는 기본 파란색을 적용하고 기본 지도 투영(`mercator`)을 사용합니다. 표준 마크 속성을 사용하여 색상과 경계선 두께를 사용자 정의할 수 있습니다. `project` 메서드를 사용하여 우리만의 지도 투영을 추가할 수도 있습니다:

In [8]:
alt.Chart(alt.topo_feature(world, 'countries')).mark_geoshape(
    fill='#2a1d0c', stroke='#706545', strokeWidth=0.5
).project(
    type='mercator'
)

alt.Chart(...)

기본적으로 Altair는 모든 데이터가 차트의 너비와 높이 내에 들어맞도록 투영을 자동으로 조정합니다. 또한 투영 설정을 사용자 정의하기 위해 `scale`(확대 레벨) 및 `translate`(이동)와 같은 투영 매개변수를 지정할 수 있습니다. 여기서는 유럽에 집중하도록 `scale` 및 `translate` 매개변수를 조정합니다:

In [9]:
alt.Chart(alt.topo_feature(world, 'countries')).mark_geoshape(
    fill='#2a1d0c', stroke='#706545', strokeWidth=0.5
).project(
    type='mercator', scale=400, translate=[100, 550]
)

alt.Chart(...)

*이 배율에서 데이터의 110m 해상도가 어떻게 나타나는지 주목하세요. 더 상세한 해안선과 경계를 보려면 더 미세한 기하학적 구조가 포함된 입력 파일이 필요합니다. `scale` 및 `translate` 매개변수를 조정하여 지도를 다른 지역에 집중시켜 보세요!*

지금까지 우리 지도는 국가만 보여주었습니다. `layer` 연산자를 사용하여 여러 지도 요소를 결합할 수 있습니다. Altair에는 추가 지도 레이어를 위한 데이터를 생성하는 데 사용할 수 있는 *데이터 생성기(data generators)*가 포함되어 있습니다:

- 구(sphere) 생성기(`{'sphere': True}`)는 지구 전체 구에 대한 GeoJSON 표현을 제공합니다. 배경 레이어로 지구의 모양을 채우는 추가 `geoshape` 마크를 만들 수 있습니다.
- 경위도망(graticule) 생성기(`{'graticule': ...}`)는 위도와 경도 선으로 형성된 격자인 *경위도망*을 나타내는 GeoJSON 피처를 생성합니다. 기본 경위도망은 위도 ±80° 사이에서 10°마다 자오선과 평행선이 있습니다. 극지방의 경우 90°마다 자오선이 있습니다. 이러한 설정은 `stepMinor` 및 `stepMajor` 속성을 사용하여 사용자 정의할 수 있습니다.

구, 경위도망 및 국가 마크를 재사용 가능한 지도 사양으로 레이어화해 보겠습니다:

In [10]:
map = alt.layer(
    # 지구의 구를 기본 레이어로 사용
    alt.Chart({'sphere': True}).mark_geoshape(
        fill='#e6f3ff'
    ),
    # 지리적 참조선을 위해 경위도망 추가
    alt.Chart({'graticule': True}).mark_geoshape(
        stroke='#ffffff', strokeWidth=1
    ),
    # 그리고 세계의 국가들 추가
    alt.Chart(alt.topo_feature(world, 'countries')).mark_geoshape(
        fill='#2a1d0c', stroke='#706545', strokeWidth=0.5
    )
).properties(
    width=600,
    height=400
)

원하는 투영법으로 지도를 확장하고 결과를 그릴 수 있습니다. 여기서는 [내추럴 어스 투영(Natural Earth projection)](https://en.wikipedia.org/wiki/Natural_Earth_projection)을 적용합니다. *sphere* 레이어는 연한 파란색 배경을 제공하고, *graticule* 레이어는 흰색 지리 참조선을 제공합니다.

In [11]:
map.project(
    type='naturalEarth1', scale=110, translate=[300, 200]
).configure_view(stroke=None)

alt.LayerChart(...)

## 점 지도 (Point Maps)

GeoJSON 또는 TopoJSON 파일에서 제공하는 *기하학적* 데이터 외에도, 많은 테이블 형식 데이터 세트에는 `경도` 및 `위도` 좌표 필드 또는 국가 이름, 주 이름, 우편 번호 등과 같은 지리적 지역에 대한 참조 형식의 지리 정보가 포함되어 있습니다. 이러한 정보는 [지오코딩 서비스](https://ko.wikipedia.org/wiki/%EC%A7%80%EC%98%A4%EC%BD%94%EB%94%A9)를 사용하여 좌표로 매핑될 수 있습니다. 어떤 경우에는 위치 데이터가 충분히 풍부하여 데이터 포인트만 투영해도 의미 있는 패턴을 볼 수 있습니다 &mdash; 기본 지도가 필요하지 않습니다!

미국의 5자리 우편번호 데이터 세트를 살펴보겠습니다. 여기에는 `zip_code` 필드 외에 각 우체국의 `longitude`(경도), `latitude`(위도) 좌표가 포함되어 있습니다.

In [12]:
zipcodes = data.zipcodes.url
zipcodes

'https://vega.github.io/vega-datasets/data/zipcodes.csv'

각 우체국 위치를 작은(1픽셀) `square` 마크를 사용하여 시각화할 수 있습니다. 그러나 위치를 설정하기 위해 `x` 및 `y` 채널을 사용하지 *않습니다*. *그 이유는 무엇일까요?*

지도 투영은 (`경도`, `위도`) 좌표를 (`x`, `y`) 좌표로 매핑하지만, 임의의 방식으로 수행될 수 있습니다. 예를 들어, `longitude` → `x` 및 `latitude` → `y`라는 보장이 없습니다! 대신 Altair에는 지리 좌표를 처리하기 위한 특수한 `longitude` 및 `latitude` 인코딩 채널이 포함되어 있습니다. 이러한 채널은 어떤 데이터 필드가 `longitude` 및 `latitude` 좌표에 매핑되어야 하는지 나타내고, 해당 좌표를 (`x`, `y`) 위치로 매핑하기 위해 투영을 적용합니다.

In [13]:
alt.Chart(zipcodes).mark_square(
    size=1, opacity=1
).encode(
    longitude='longitude:Q', # 'longitude'라는 이름의 필드를 경도 채널에 적용
    latitude='latitude:Q'    # 'latitude'라는 이름의 필드를 위도 채널에 적용
).project(
    type='albersUsa'
).properties(
    width=900,
    height=500
).configure_view(
    stroke=None
)

alt.Chart(...)

*우편번호만 플로팅해도 기본 지도나 추가 참조 요소 없이 미국의 윤곽을 볼 수 있고 우체국 밀도에서 의미 있는 패턴을 식별할 수 있습니다!*

우리는 지구의 실제 기하학적 구조를 약간 변형하여 알래스카와 하와이를 왼쪽 하단에 축소해서 보여주는 `albersUsa` 투영법을 사용합니다. 투영 `scale`이나 `translate` 매개변수를 지정하지 않았으므로, Altair는 시각화된 데이터에 맞게 자동으로 설정합니다.

이제 데이터 세트에 대해 더 많은 질문을 던질 수 있습니다. 예를 들어, 우편번호 할당에 어떤 규칙이나 이유가 있을까요? 이 질문을 평가하기 위해 우편번호의 첫 번째 자리를 기반으로 색상 인코딩을 추가할 수 있습니다. 먼저 첫 번째 자리를 추출하기 위해 `calculate` 변환을 추가하고, 결과를 색상 채널을 사용하여 인코딩합니다:

In [14]:
alt.Chart(zipcodes).transform_calculate(
    digit='datum.zip_code[0]'
).mark_square(
    size=2, opacity=1
).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    color='digit:N'
).project(
    type='albersUsa'
).properties(
    width=900,
    height=500
).configure_view(
    stroke=None
)

alt.Chart(...)

*특정 숫자를 확대하려면 필터 변환을 추가하여 표시되는 데이터를 제한하세요! 단일 숫자로 필터링하고 지도를 동적으로 업데이트하기 위해 [대화형 선택](https://github.com/uwdata/visualization-curriculum/blob/master/altair_interaction.ipynb)을 추가해 보세요. 그리고 숫자 값을 필터링할 때 숫자(`1`) 대신 문자열(`'1'`)을 사용해야 합니다!*

(이 예제는 Ben Fry의 고전적인 [zipdecode](https://benfry.com/zipdecode/) 시각화에서 영감을 받았습니다!)

우편번호의 *순서*가 무엇을 나타내는지 더 궁금할 수 있습니다. 이 질문을 탐구하는 한 가지 방법은 Robert Kosara의 [ZipScribble](https://eagereyes.org/zipscribble-maps/united-states) 시각화에서와 같이 각 연속된 우편번호를 `line` 마크를 사용하여 연결하는 것입니다:

In [15]:
alt.Chart(zipcodes).transform_filter(
    '-150 < datum.longitude && 22 < datum.latitude && datum.latitude < 55'
).transform_calculate(
    digit='datum.zip_code[0]'
).mark_line(
    strokeWidth=0.5
).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    color='digit:N',
    order='zip_code:O'
).project(
    type='albersUsa'
).properties(
    width=900,
    height=500
).configure_view(
    stroke=None
)

alt.Chart(...)

*이제 우편번호가 어떻게 더 작은 영역으로 클러스터링되는지 볼 수 있습니다. 이는 위치에 따른 코드의 계층적 할당을 나타내지만, 지역 클러스터 내에서 눈에 띄는 변동성도 있습니다.*

이전 지도를 주의 깊게 보셨다면 왼쪽 상단에 플로팅되는 우편번호가 있다는 것을 알아차리셨을 것입니다! 이들은 푸에르토리코나 아메리칸 사모아와 같이 미국 우편번호를 포함하지만 `albersUsa` 투영법에 의해 `null` 좌표(`0`, `0`)로 매핑되는 위치에 해당합니다. 또한 알래스카와 하와이는 연결된 선 세그먼트의 보기를 복잡하게 만들 수 있습니다. 이에 대응하여 위의 코드는 선택한 `longitude` 및 `latitude` 범위 밖의 점들을 제거하는 추가 필터를 포함합니다.

*무엇이 일어나는지 보려면 위의 필터를 제거해 보세요!*

## 기호 지도 (Symbol Maps)

이제 기본 지도와 플로팅된 데이터를 별도의 레이어로 결합해 보겠습니다. 공항과 비행 경로를 모두 고려하여 미국의 상업 비행 네트워크를 조사할 것입니다. 이를 위해 세 개의 데이터 세트가 필요합니다.
기본 지도로는 주(`states`) 또는 카운티(`counties`) 피처를 포함하는 10m 해상도의 미국용 TopoJSON 파일을 사용합니다:

In [16]:
usa = data.us_10m.url
usa

'https://vega.github.io/vega-datasets/data/us-10m.json'

공항의 경우, 각 공항의 `경도` 및 `위도` 좌표와 `iata` 공항 코드(예: [시애틀-타코마 국제공항](https://ko.wikipedia.org/wiki/%EC%8B%9C%EC%95%A0%ED%8B%80_%ED%83%80%EC%BD%94%EB%A7%88_%EA%B5%AD%EC%A0%9C%EA%B3%B5%ED%95%AD)의 경우 `'SEA'`)가 포함된 필드가 있는 데이터 세트를 사용합니다.

In [17]:
airports = data.airports.url
airports

'https://vega.github.io/vega-datasets/data/airports.csv'

마지막으로, 해당 공항의 IATA 코드가 포함된 `origin`(출발지) 및 `destination`(목적지) 필드가 있는 비행 경로 데이터 세트를 사용합니다:

In [18]:
flights = data.flights_airport.url
flights

'https://vega.github.io/vega-datasets/data/flights-airport.csv'

`albersUsa` 투영법을 사용하여 기본 지도를 만들고, 각 공항에 대해 `circle` 마크를 플로팅하는 레이어를 추가해 보겠습니다:

In [19]:
alt.layer(
    alt.Chart(alt.topo_feature(usa, 'states')).mark_geoshape(
        fill='#ddd', stroke='#fff', strokeWidth=1
    ),
    alt.Chart(airports).mark_circle(size=9).encode(
        latitude='latitude:Q',
        longitude='longitude:Q',
        tooltip='iata:N'
    )
).project(
    type='albersUsa'
).properties(
    width=900,
    height=500
).configure_view(
    stroke=None
)

alt.LayerChart(...)

*공항이 정말 많네요! 분명히 모든 공항이 주요 허브는 아닐 것입니다.*

우편번호 데이터 세트와 마찬가지로, 공항 데이터에도 미국 본토 이외의 지점이 포함되어 있습니다. 그래서 다시 한번 왼쪽 상단에 점들이 보입니다. 이러한 지점들을 필터링하고 싶을 수 있지만, 그러기 위해서는 먼저 그것들에 대해 더 알아야 합니다.

*위의 지도 투영법을 `albers`로 업데이트하여 &ndash; `albersUsa`의 독특한 동작을 피하고 &ndash; 이러한 추가 지점들의 실제 위치가 드러나도록 해보세요!*

이제 모든 공항을 차별 없이 보여주는 대신, 각 공항에서 출발하는 총 노선 수를 고려하여 주요 허브를 식별해 보겠습니다. `routes` 데이터 세트를 기본 데이터 소스로 사용하겠습니다. 여기에는 각 `origin`(출발지) 공항에 대한 노선 수를 합산하기 위해 집계할 수 있는 비행 노선 목록이 포함되어 있습니다.

그러나 `routes` 데이터 세트에는 공항의 *위치*가 포함되어 있지 않습니다! `routes` 데이터에 위치 정보를 보강하려면 새로운 데이터 변환인 `lookup`이 필요합니다. `lookup` 변환은 기본 데이터 세트의 필드 값을 가져와 다른 테이블에서 관련 정보를 찾기 위한 *키(key)*로 사용합니다. 이 경우, `routes` 데이터 세트의 `origin` 공항 코드를 `airports` 데이터 세트의 `iata` 필드와 일치시킨 다음 해당 `latitude`(위도) 및 `longitude`(경도) 필드를 추출하려고 합니다.

In [20]:
alt.layer(
    alt.Chart(alt.topo_feature(usa, 'states')).mark_geoshape(
        fill='#ddd', stroke='#fff', strokeWidth=1
    ),
    alt.Chart(flights).mark_circle().transform_aggregate(
        groupby=['origin'],
        routes='count()'
    ).transform_lookup(
        lookup='origin',
        from_=alt.LookupData(data=airports, key='iata',
                             fields=['state', 'latitude', 'longitude'])
    ).transform_filter(
        'datum.state !== "PR" && datum.state !== "VI"'
    ).encode(
        latitude='latitude:Q',
        longitude='longitude:Q',
        tooltip=['origin:N', 'routes:Q'],
        size=alt.Size('routes:Q', scale=alt.Scale(range=[0, 1000]), legend=None),
        order=alt.Order('routes:Q', sort='descending')
    )
).project(
    type='albersUsa'
).properties(
    width=900,
    height=500
).configure_view(
    stroke=None
)

alt.LayerChart(...)

*어느 미국 공항이 가장 많은 출발 노선을 가지고 있나요?*

이제 공항을 볼 수 있으므로, 비행 교통 네트워크의 구조를 더 잘 이해하기 위해 공항과 상호작용하고 싶을 수 있습니다. 각 끝점의 좌표를 검색하기 위해 두 개의 `lookup` 변환이 필요한 `origin`(출발) 공항에서 `destination`(도착) 공항으로의 경로를 나타내는 `rule` 마크 레이어를 추가할 수 있습니다. 또한 현재 선택된 공항에서 출발하는 노선만 표시되도록 이러한 노선을 필터링하기 위해 `single` 선택을 사용할 수 있습니다.

*위의 정적 지도에서 시작하여 대화형 버전을 만들 수 있습니까? 아래 코드를 건너뛰고 먼저 대화형 지도와 상호작용해 보면서, 스스로 어떻게 만들 수 있을지 생각해 보세요!*

In [21]:
# 출발 공항에 대한 대화형 선택
# 마우스 커서와 가장 가까운 공항 선택
origin = alt.selection_single(
    on='mouseover', nearest=True,
    fields=['origin'], empty='none'
)

# 조회를 위한 공유 데이터 참조
foreign = alt.LookupData(data=airports, key='iata',
                         fields=['latitude', 'longitude'])
    
alt.layer(
    # 미국의 기본 지도
    alt.Chart(alt.topo_feature(usa, 'states')).mark_geoshape(
        fill='#ddd', stroke='#fff', strokeWidth=1
    ),
    # 선택된 출발 공항에서 목적지 공항으로의 노선
    alt.Chart(flights).mark_rule(
        color='#000', opacity=0.35
    ).transform_filter(
        origin # 선택된 출발지만 필터링
    ).transform_lookup(
        lookup='origin', from_=foreign # 출발지 위도/경도
    ).transform_lookup(
        lookup='destination', from_=foreign, as_=['lat2', 'lon2'] # 목적지 위도/경도
    ).encode(
        latitude='latitude:Q',
        longitude='longitude:Q',
        latitude2='lat2',
        longitude2='lon2',
    ),
    # 노선 수에 따라 공항 크기 조절
    # 1. flights-airport 데이터 세트 집계
    # 2. airports 데이터 세트에서 위치 데이터 조회
    # 3. 푸에르토리코(PR) 및 버진 아일랜드(VI) 제거
    alt.Chart(flights).mark_circle().transform_aggregate(
        groupby=['origin'],
        routes='count()'
    ).transform_lookup(
        lookup='origin',
        from_=alt.LookupData(data=airports, key='iata',
                             fields=['state', 'latitude', 'longitude'])
    ).transform_filter(
        'datum.state !== "PR" && datum.state !== "VI"'
    ).add_selection(
        origin
    ).encode(
        latitude='latitude:Q',
        longitude='longitude:Q',
        tooltip=['origin:N', 'routes:Q'],
        size=alt.Size('routes:Q', scale=alt.Scale(range=[0, 1000]), legend=None),
        order=alt.Order('routes:Q', sort='descending') # 작은 원을 위에 배치
    )
).project(
    type='albersUsa'
).properties(
    width=900,
    height=500
).configure_view(
    stroke=None
)

alt.LayerChart(...)

*지도를 마우스로 가리켜 비행 네트워크를 조사해 보세요!*

## 단계 구분도 (Choropleth Maps)

[*단계 구분도(choropleth map)*](https://ko.wikipedia.org/wiki/%EB%8B%A8%EA%B3%84_%EA%B5%AC%EB%B6%84%EB%8F%84)는 데이터 값을 시각화하기 위해 색이 칠해지거나 질감이 있는 영역을 사용합니다. 사람들은 원의 면적 사이의 비례적 차이를 추정하는 것이 색상 음영 사이의 차이를 추정하는 것보다 더 나은 경향이 있기 때문에, 크기가 조정된 기호 지도가 종종 더 정확하게 읽힙니다. 그럼에도 불구하고 단계 구분도는 실제로 널리 사용되며, 특히 기호가 너무 많아 지각적으로 압도될 때 유용합니다.

예를 들어, 미국에는 50개의 주만 있지만, 그 주 안에는 수천 개의 카운티가 있습니다. 2008년 경기 침체 당시의 카운티별 실업률 단계 구분도를 만들어 보겠습니다. 어떤 경우에는 입력 GeoJSON 또는 TopoJSON 파일에 우리가 직접 시각화할 수 있는 통계 데이터가 포함되어 있을 수 있습니다. 하지만 이 경우에는 두 개의 파일이 있습니다: 카운티 경계 피처(`usa`)가 포함된 TopoJSON 파일과 실업률 통계가 포함된 별도의 텍스트 파일입니다:

In [22]:
unemp = data.unemployment.url
unemp

'https://vega.github.io/vega-datasets/data/unemployment.tsv'

데이터 소스를 통합하려면 TopoJSON 기반의 `geoshape` 데이터에 실업률을 보강하기 위해 다시 한번 `lookup` 변환을 사용해야 합니다. 그런 다음 조회된 `rate`(실업률) 필드에 대해 `color` 인코딩을 포함하는 지도를 만들 수 있습니다.

In [23]:
alt.Chart(alt.topo_feature(usa, 'counties')).mark_geoshape(
    stroke='#aaa', strokeWidth=0.25
).transform_lookup(
    lookup='id', from_=alt.LookupData(data=unemp, key='id', fields=['rate'])
).encode(
    alt.Color('rate:Q',
              scale=alt.Scale(domain=[0, 0.3], clamp=True), 
              legend=alt.Legend(format='%')),
    alt.Tooltip('rate:Q', format='.0%')
).project(
    type='albersUsa'
).properties(
    width=900,
    height=500
).configure_view(
    stroke=None
)

alt.Chart(...)

*카운티별 실업률을 조사해 보세요. 미시간주의 높은 수치는 자동차 산업과 관련이 있을 수 있습니다. [그레이트플레인스(Great Plains)](https://ko.wikipedia.org/wiki/%EA%B7%B8%EB%A0%88%EC%9D%B4%ED%8A%B8%ED%94%8C%EB%A0%88%EC%9D%B8%EC%8A%A4)와 산악 지역의 주들은 낮고 **동시에** 높은 실업률을 보입니다. 이러한 변동이 의미가 있는 것일까요, 아니면 [낮은 표본 크기의 인위적인 결과(artifact)](https://medium.com/@uwdata/surprise-maps-showing-the-unexpected-e92b67398865)일까요? 더 자세히 탐색하려면 상단 스케일 도메인을 변경하여(예: `0.2`) 색상 매핑을 조정해 보세요.*

단계 구분도에서 핵심적인 관심사는 색상 선택입니다. 위에서 우리는 히트맵을 위해 Altair의 기본값인 `'yellowgreenblue'` 체계를 사용했습니다. 아래에서는 휘도(luminance)만 변하는 *단일 색조 순차(single-hue sequential)* 체계(`teals`), 휘도와 색조가 모두 변하는 *다중 색조 순차(multi-hue sequential)* 체계(`viridis`), 그리고 흰색 중간 지점을 사용하는 *발산(diverging)* 체계(`blueorange`)를 포함한 다른 체계들을 비교합니다:

In [24]:
# 제공된 색상 체계에 대한 지도 사양을 생성하는 유틸리티 함수
def map_(scheme):
    return alt.Chart().mark_geoshape().project(type='albersUsa').encode(
        alt.Color('rate:Q', scale=alt.Scale(scheme=scheme), legend=None)
    ).properties(width=305, height=200)

alt.hconcat(
    map_('tealblues'), map_('viridis'), map_('blueorange'),
    data=alt.topo_feature(usa, 'counties')
).transform_lookup(
    lookup='id', from_=alt.LookupData(data=unemp, key='id', fields=['rate'])
).configure_view(
    stroke=None
).resolve_scale(
    color='independent'
)

alt.HConcatChart(...)

*어떤 색상 체계가 더 효과적이라고 생각하시나요? 그 이유는 무엇일까요? [Vega 색상 체계 문서](https://vega.github.io/vega/docs/schemes/)에 설명된 다른 사용 가능한 체계를 사용하도록 위의 지도를 수정해 보세요.*

## 지도 투영 (Cartographic Projections)

이제 지도를 만드는 데 어느 정도 경험이 쌓였으니, 지도 투영에 대해 더 자세히 살펴보겠습니다. [위키백과](https://ko.wikipedia.org/wiki/%EC%A7%80%EB%8F%84_%ED%82%AC%EC%98%81%EB%B2%95)에서 설명하듯이,

> *모든 지도 투영법은 필연적으로 어떤 방식으로든 표면을 왜곡합니다. 지도의 목적에 따라 어떤 왜곡은 허용되고 어떤 왜곡은 허용되지 않습니다. 따라서 구와 유사한 물체의 일부 특성을 보존하면서 다른 특성을 희생하기 위해 다양한 지도 투영법이 존재합니다.*

우리가 고려하고 싶은 몇 가지 특성은 다음과 같습니다:

- *면적(Area)*: 투영법이 지역의 크기를 왜곡합니까?
- *방위(Bearing)*: 직선이 일정한 이동 방향에 해당합니까?
- *거리(Distance)*: 길이가 같은 선이 지구상의 동일한 거리에 해당합니까?
- *모양(Shape)*: 투영법이 점들 사이의 공간 관계(각도)를 보존합니까?

따라서 적절한 투영법을 선택하는 것은 지도의 사용 사례에 달려 있습니다. 예를 들어, 토지 이용을 평가하고 토지의 범위가 중요하다면 면적 보존 투영법을 선택할 수 있습니다. 지진에서 발생하는 충격파를 시각화하고 싶다면 지진의 진앙에 지도를 맞추고 그 지점으로부터의 거리를 보존할 수 있습니다. 또는 항해를 돕고 싶다면 방위와 모양의 보존이 더 중요할 수 있습니다.

우리는 또한 *투영면(projection surface)*의 관점에서 투영법을 분류할 수 있습니다. 예를 들어, 원통 도법은 구의 표면 지점을 둘러싼 원통에 투영합니다. 그런 다음 "펼쳐진" 원통이 우리의 지도가 됩니다. 아래에서 더 자세히 설명하겠지만, 대안으로 원뿔의 표면(원뿔 도법)이나 평평한 평면(방위 도법)에 직접 투영할 수도 있습니다.

*먼저 다양한 투영법과 상호작용하며 직관을 쌓아봅시다! **[온라인 Vega-Lite 지도 투영법 노트북](https://observablehq.com/@vega/vega-lite-cartographic-projections)**을 열어보세요. 해당 페이지의 컨트롤을 사용하여 투영법을 선택하고 `scale`(확대/축소) 및 x/y 이동(이동)과 같은 투영 매개변수를 탐색해 보세요. 회전([yaw, pitch, roll](https://ko.wikipedia.org/wiki/%ED%95%AD%EA%B3%B5%EA%B8%B0_%EC%A3%BC%EC%B6%95)) 컨트롤은 투영되는 표면에 대한 지구의 방향을 결정합니다.*

### 특정 투영 유형 둘러보기 (A Tour of Specific Projection Types)

[**원통 도법(Cylindrical projections)**](https://ko.wikipedia.org/wiki/%EC%A7%80%EB%8F%84_%ED%82%AC%EC%98%81%EB%B2%95#%EC%9B%90%ED%86%B5_%EB%8F%84%EB%B2%95)은 구를 둘러싼 원통에 투영한 다음 원통을 펼칩니다. 원통의 주축이 남북 방향으로 정렬되면 자오선은 직선으로 매핑됩니다. [의원통 도법(Pseudo-cylindrical)](https://ko.wikipedia.org/wiki/%EC%A7%80%EB%8F%84_%ED%82%AC%EC%98%81%EB%B2%95#%EC%9D%98%EC%9B%90%ED%86%B5_%EB%8F%84%EB%B2%95)은 중앙 자오선을 직선으로 표현하고, 다른 자오선들은 중심에서 멀어지며 "휘어지게" 표현합니다.

In [25]:
minimap = map.properties(width=225, height=225)
alt.hconcat(
    minimap.project(type='equirectangular').properties(title='equirectangular'),
    minimap.project(type='mercator').properties(title='mercator'),
    minimap.project(type='transverseMercator').properties(title='transverseMercator'),
    minimap.project(type='naturalEarth1').properties(title='naturalEarth1')
).properties(spacing=10).configure_view(stroke=None)

alt.HConcatChart(...)

- [정거 원통 도법(Equirectangular)](https://ko.wikipedia.org/wiki/%EC%A0%95%EA%B1%B0_%EC%9B%90%ED%86%B5_%EB%8F%84%EB%B2%95) (`equirectangular`): `lat`(위도), `lon`(경도) 좌표 값을 직접 스케일링합니다.
- [메르카토르 도법(Mercator)](https://ko.wikipedia.org/wiki/%EB%A9%94%EB%A5%B4%EC%B9%B4%ED%86%A0%EB%A5%B4_%EB%8F%84%EB%B2%95) (`mercator`): `lon`을 직접 사용하여 원통에 투영하지만, `lat`에는 비선형 변환을 적용합니다. 직선은 일정한 나침반 방위([항정선(rhumb lines)](https://ko.wikipedia.org/wiki/%ED%95%AD%EC%A0%95%EC%84%A0))를 보존하므로 항해에 적합합니다. 그러나 극지방에 가까울수록 면적이 크게 왜곡될 수 있습니다.
- [횡축 메르카토르 도법(Transverse Mercator)](https://ko.wikipedia.org/wiki/%ED%9A%A1%EC%B6%95_%EB%A9%94%EB%A5%B4%EC%B9%B4%ED%86%A0%EB%A5%B4_%EB%8F%84%EB%B2%95) (`transverseMercator`): 메르카토르 도법이지만 둘러싼 원통을 가로축으로 회전시킨 형태입니다. 표준 메르카토르 도법이 적도를 따라 정확도가 가장 높은 반면, 횡축 메르카토르 도법은 중앙 자오선을 따라 가장 정확합니다.
- [내추럴 어스(Natural Earth)](https://en.wikipedia.org/wiki/Natural_Earth_projection) (`naturalEarth1`): 지구 전체를 하나의 뷰로 보여주기 위해 설계된 의원통 도법입니다.
<br/><br/>

[**원뿔 도법(Conic projections)**](https://ko.wikipedia.org/wiki/%EC%A7%80%EB%8F%84_%ED%82%AC%EC%98%81%EB%B2%95#%EC%9B%90%EB%BD%90_%EB%8F%84%EB%B2%95)은 구를 원뿔에 투영한 다음, 원뿔을 평면에 펼칩니다. 원뿔 도법은 원뿔이 지구와 교차하는 위치를 결정하는 두 개의 *표준 평행선*에 의해 설정됩니다.

In [26]:
minimap = map.properties(width=180, height=130)
alt.hconcat(
    minimap.project(type='conicEqualArea').properties(title='conicEqualArea'),
    minimap.project(type='conicEquidistant').properties(title='conicEquidistant'),
    minimap.project(type='conicConformal', scale=35, translate=[90,65]).properties(title='conicConformal'),
    minimap.project(type='albers').properties(title='albers'),
    minimap.project(type='albersUsa').properties(title='albersUsa')
).properties(spacing=10).configure_view(stroke=None)

alt.HConcatChart(...)

- [앨버스 정적 원뿔 도법(Conic Equal Area)](https://ko.wikipedia.org/wiki/%EC%95%A8%EB%B2%84%EC%8A%A4_%EC%A0%95%EC%A0%81_%EC%9B%90%EB%BD%90_%EB%8F%84%EB%B2%95) (`conicEqualArea`): 면적을 보존하는 원뿔 도법입니다. 모양과 거리는 보존되지 않지만 표준 평행선 내에서 대략적으로 정확합니다.
- [정거 원뿔 도법(Conic Equidistant)](https://ko.wikipedia.org/wiki/%EC%A0%95%EA%B1%B0_%EC%9B%90%EB%BD%90_%EB%8F%84%EB%B2%95) (`conicEquidistant`): 자오선과 표준 평행선을 따라 거리를 보존하는 원뿔 도법입니다.
- [람베르트 정각 원뿔 도법(Conic Conformal)](https://ko.wikipedia.org/wiki/%EB%9E%8C%EB%B2%A0%EB%A5%B4%ED%8A%B8_%EC%A0%95%EA%B0%81_%EC%9B%90%EB%BD%90_%EB%8F%84%EB%B2%95) (`conicConformal`): 모양(국소적 각도)을 보존하지만 면적이나 거리는 보존하지 않는 원뿔 도법입니다.
- [앨버스 도법(Albers)](https://ko.wikipedia.org/wiki/%EC%95%A8%EB%B2%84%EC%8A%A4_%EC%A0%95%EC%A0%81_%EC%9B%90%EB%BD%90_%EB%8F%84%EB%B2%95) (`albers`): 미국 지도를 만드는 데 최적화된 표준 평행선을 가진 앨버스 정적 원뿔 도법의 변형입니다.
- [앨버스 USA(Albers USA)](https://en.wikipedia.org/wiki/Albers_projection) (`albersUsa`): 미국의 50개 주를 위한 하이브리드 투영법입니다. 이 투영법은 미국 본토, 알래스카, 하와이에 대해 서로 다른 매개변수를 가진 세 개의 앨버스 투영법을 결합합니다.
<br/><br/>

[**방위 도법(Azimuthal projections)**](https://ko.wikipedia.org/wiki/%EC%A7%80%EB%8F%84_%ED%82%AC%EC%98%81%EB%B2%95#%EB%B0%A9%EC%9C%84_%EB%8F%84%EB%B2%95)은 구를 평면에 직접 투영합니다.

In [27]:
minimap = map.properties(width=180, height=180)
alt.hconcat(
    minimap.project(type='azimuthalEqualArea').properties(title='azimuthalEqualArea'),
    minimap.project(type='azimuthalEquidistant').properties(title='azimuthalEquidistant'),
    minimap.project(type='orthographic').properties(title='orthographic'),
    minimap.project(type='stereographic').properties(title='stereographic'),
    minimap.project(type='gnomonic').properties(title='gnomonic')
).properties(spacing=10).configure_view(stroke=None)

alt.HConcatChart(...)

- [람베르트 정적 방위 도법(Azimuthal Equal Area)](https://ko.wikipedia.org/wiki/%EB%9E%8C%EB%B2%A0%EB%A5%B4%ED%8A%B8_%EC%A0%91%ED%8F%89%EB%A9%B4_%EC%A0%95%EC%A0%81_%EB%8F%84%EB%B2%95) (`azimuthalEqualArea`): 지구의 모든 부분에서 면적을 정확하게 투영하지만 모양(국소적 각도)은 보존하지 않습니다.
- [정거 방위 도법(Azimuthal Equidistant)](https://ko.wikipedia.org/wiki/%EC%A0%95%EA%B1%B0_%EB%B0%A9%EC%9C%84_%EB%8F%84%EB%B2%95) (`azimuthalEquidistant`): 투영 중심에서 지구상의 다른 모든 점까지의 비례적인 거리를 보존합니다.
- [정사 도법(Orthographic)](https://ko.wikipedia.org/wiki/%EC%A0%95%EC%82%AC_%EB%8F%84%EB%B2%95) (`orthographic`): 보이는 반구를 먼 평면에 투영합니다. 우주에서 본 지구의 모습과 비슷합니다.
- [평사 도법(Stereographic)](https://ko.wikipedia.org/wiki/%ED%8F%89%EC%82%AC_%EB%8F%84%EB%B2%95) (`stereographic`): 모양은 보존하지만 면적이나 거리는 보존하지 않습니다.
- [심사 도법(Gnomonic)](https://ko.wikipedia.org/wiki/%EC%8B%AC%EC%82%AC_%EB%8F%84%EB%B2%95) (`gnomonic`): 구의 표면을 접평면에 직접 투영합니다. 지구상의 [대권(Great circles)](https://ko.wikipedia.org/wiki/%EB%8C%80%EA%B6%8C)이 직선으로 투영되어 점 사이의 최단 경로를 보여줍니다.
<br/><br/>

## 코다: 지리적 데이터 다루기 (Coda: Wrangling Geographic Data)

위의 예제들은 모두 기하학적(TopoJSON) 및 테이블 형식(공항, 실업률) 데이터를 포함하여 vega-datasets 컬렉션에서 가져온 것입니다. 지리적 시각화를 시작할 때 흔히 겪는 어려움은 작업에 필요한 데이터를 수집하는 것입니다. [미국 지질조사국(USGS)](https://www.usgs.gov/products/data/all-data) 및 [미국 인구조사국(U.S. Census Bureau)](https://www.census.gov/data/datasets.html)과 같은 서비스를 포함하여 수많은 데이터 제공자가 존재합니다.

많은 경우 지리적 구성 요소가 포함된 기존 데이터를 가지고 있을 수 있지만, 추가적인 측정값이나 기하학적 구조가 필요할 수 있습니다. 시작하는 데 도움이 되도록 다음은 하나의 워크플로입니다:

1. [Natural Earth Data](http://www.naturalearthdata.com/downloads/)를 방문하여 관심 있는 지역과 해상도의 데이터를 탐색하고 선택합니다. 해당 zip 파일을 다운로드합니다.
2. [MapShaper](https://mapshaper.org/)로 이동하여 다운로드한 zip 파일을 페이지에 끌어다 놓습니다. 원하는 대로 데이터를 수정하고 생성된 TopoJSON 또는 GeoJSON 파일을 "내보내기(Export)"합니다.
3. MapShaper에서 내보낸 데이터를 Altair에서 사용하기 위해 로드합니다!

물론 지리적 데이터를 다루기 위한 오픈 소스 및 상용 도구가 많이 존재합니다. 지리 데이터 랭글링 및 지도 생성에 대한 자세한 내용은 Mike Bostock의 [명령행 지도 제작(Command-Line Cartography)](https://medium.com/@mbostock/command-line-cartography-part-1-897aa8f8ca2c) 튜토리얼 시리즈를 참조하세요.

## 요약 (Summary)

지금까지 우리는 지도 제작의 세계에 발을 살짝 담갔을 뿐입니다. *(단 하나의 노트북으로 수세기에 걸친 학문을 다 배울 수 있을 거라고 기대하시진 않으셨겠죠?)* 예를 들어, [*카토그램(cartograms)*](https://ko.wikipedia.org/wiki/%EC%B9%B4%ED%86%A0%EA%B7%B8%EB%9E%A8)이나 Imhof의 계몽적인 저서 [*Cartographic Relief Presentation*](https://books.google.com/books?id=cVy1Ms43fFYC)에서와 같은 *지형(topography)* 전달과 같은 주제는 다루지 않았습니다. 그럼에도 불구하고 이제 여러분은 다양한 지리 시각화를 만들 수 있는 충분한 준비가 되었을 것입니다. 더 자세한 내용은 MacEachren의 저서 [*How Maps Work: Representation, Visualization, and Design*](https://books.google.com/books?id=xhAvN3B0CkUC)이 데이터 시각화의 관점에서 지도 제작에 대한 가치 있는 개요를 제공합니다.